In [2]:
import pandas as pd
import numpy as np
# Read the truck traffic flow data
df=pd.read_csv('01_Trucktrafficflow.csv')


In [16]:
# List of all regions in the netherlands
regions = [
    "Delfzijl en omgeving", "Oost-Groningen", "Overig Groningen", "Zuidoost-Friesland",
    "Noord-Friesland", "Zuidwest-Friesland", "Noord-Drenthe", "Zuidoost-Drenthe",
    "Zuidwest-Drenthe", "Noord-Overijssel", "Zuidwest-Overijssel", "Twente",
    "Veluwe", "Zuidwest-Gelderland", "Achterhoek", "Arnhem/Nijmegen", "Flevoland",
    "Kop van Noord-Holland", "IJmond", "Zaanstreek", "Het Gooi en Vechtstreek",
    "Alkmaar en omgeving", "Agglomeratie Haarlem", "Groot-Amsterdam",
    "Zeeuwsch-Vlaanderen", "Overig Zeeland", "Utrecht", "Agglomeratie ’s-Gravenhage",
    "Delft en Westland", "Agglomeratie Leiden en Bollenstreek", "Zuidoost-Zuid-Holland",
    "Oost-Zuid-Holland", "Groot-Rijnmond", "West-Noord-Brabant", "Zuidoost-Noord-Brabant",
    "Midden-Noord-Brabant", "Noordoost-Noord-Brabant", "Noord-Limburg",
    "Midden-Limburg", "Zuid-Limburg"
]



# Filter the dataframe to only include rows where the origin region is in the Netherlands
filtered_df = df.loc[
    df["Name_origin_region"].isin(regions),
    ["Name_origin_region", "Name_destination_region", "Total_distance", "Traffic_flow_tons_2019", "Edge_path_E_road"]
]
filtered_df.head()

,Name_origin_region,Name_destination_region,Total_distance,Traffic_flow_tons_2019,Edge_path_E_road
1087774,Oost-Groningen,Mittelburgenland,1145.0,17.0,"[1034974, 1008535, 1008552, 1000001, 1008851, ..."
1087775,Oost-Groningen,Nordburgenland,1121.0,51.0,"[1008555, 2500448, 1008540, 1008817, 1008826, ..."
1087776,Oost-Groningen,Sudburgenland,1243.0,17.0,"[1008563, 1310426, 1009011, 1035597, 1008566, ..."
1087777,Oost-Groningen,Mostviertel-Eisenwurzen,1085.0,187.0,"[1008584, 1008595, 1008621, 1035056, 1008618, ..."
1087778,Oost-Groningen,Niederosterreich-Sud,1164.0,221.0,"[1008654, 1008675, 1000017, 1035287, 1008678, ..."


In [17]:

# Coordinates of sources
points = np.array([
    [4.5967195, 51.832247,],  # Kijfhoek
    [4.6058241, 51.674466,],  # Moerdijk
    [3.6938758, 51.448118,],  # Sloe
    [4.8997765, 52.379106,],  # Amsterdam
])

point_names = ["Kijfhoek", "Moerdijk", "Sloe", "Amsterdam"]

# Load region data
Reg = pd.read_csv("02_NUTS-3-Regions.csv")

regions = [
    "Delfzijl en omgeving", "Oost-Groningen", "Overig Groningen", "Zuidoost-Friesland",
    "Noord-Friesland", "Zuidwest-Friesland", "Noord-Drenthe", "Zuidoost-Drenthe",
    "Zuidwest-Drenthe", "Noord-Overijssel", "Zuidwest-Overijssel", "Twente",
    "Veluwe", "Zuidwest-Gelderland", "Achterhoek", "Arnhem/Nijmegen", "Flevoland",
    "Kop van Noord-Holland", "IJmond", "Zaanstreek", "Het Gooi en Vechtstreek",
    "Alkmaar en omgeving", "Agglomeratie Haarlem", "Groot-Amsterdam",
    "Zeeuwsch-Vlaanderen", "Overig Zeeland", "Utrecht", "Agglomeratie ’s-Gravenhage",
    "Delft en Westland", "Agglomeratie Leiden en Bollenstreek", "Zuidoost-Zuid-Holland",
    "Oost-Zuid-Holland", "Groot-Rijnmond", "West-Noord-Brabant", "Zuidoost-Noord-Brabant",
    "Midden-Noord-Brabant", "Noordoost-Noord-Brabant", "Noord-Limburg",
    "Midden-Limburg", "Zuid-Limburg"
]

# Create a new DataFrame with only the regions in the Netherlands and their geometric centers
Reg_df = Reg.loc[
    Reg["Name"].isin(regions),
    ["Name", "Geometric_center", "Geometric_center_X", "Geometric_center_Y"]
].copy()

# Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


# Compute distances from geometric centers of regions to each source
distances = []

for lat, lon in points:
    dist = haversine(
        Reg_df["Geometric_center_X"].values,  # lat
        Reg_df["Geometric_center_Y"].values,  # lon
        lat,
        lon
    )
    distances.append(dist)

distances = np.array(distances)  # shape: (num_points, num_regions)


# Find closest source
closest_idx = distances.argmin(axis=0)

Reg_df["min_distance_km"] = distances.min(axis=0)
Reg_df["closest_point"] = [point_names[i] for i in closest_idx]



In [18]:
# Merge region data with closest source information with truck flow data
filtered_df = filtered_df.merge(
    Reg_df[['Name', 'min_distance_km', 'closest_point']],
    left_on='Name_origin_region',
    right_on='Name',
    how='left'
)
filtered_df


,Name_origin_region,Name_destination_region,Total_distance,Traffic_flow_tons_2019,Edge_path_E_road,Name,min_distance_km,closest_point
0,Oost-Groningen,Mittelburgenland,1145.0,17.0,"[1034974, 1008535, 1008552, 1000001, 1008851, ...",Oost-Groningen,246.267106,Amsterdam
1,Oost-Groningen,Nordburgenland,1121.0,51.0,"[1008555, 2500448, 1008540, 1008817, 1008826, ...",Oost-Groningen,246.267106,Amsterdam
2,Oost-Groningen,Sudburgenland,1243.0,17.0,"[1008563, 1310426, 1009011, 1035597, 1008566, ...",Oost-Groningen,246.267106,Amsterdam
3,Oost-Groningen,Mostviertel-Eisenwurzen,1085.0,187.0,"[1008584, 1008595, 1008621, 1035056, 1008618, ...",Oost-Groningen,246.267106,Amsterdam
4,Oost-Groningen,Niederosterreich-Sud,1164.0,221.0,"[1008654, 1008675, 1000017, 1035287, 1008678, ...",Oost-Groningen,246.267106,Amsterdam
...,...,...,...,...,...,...,...,...
52187,Zuid-Limburg,Kabardin-Balkar,4562.0,357.0,"[1069947, 1069948, 1069949, 1069951, 1069960, ...",Zuid-Limburg,163.663380,Moerdijk
52188,Zuid-Limburg,Karachay-Cherkess,4759.0,357.0,"[1069947, 1069948, 1069949, 1069951, 1069960, ...",Zuid-Limburg,163.663380,Moerdijk
52189,Zuid-Limburg,North Ossetia,4480.0,357.0,"[1069947, 1069948, 1069949, 1069951, 1069960, ...",Zuid-Limburg,163.663380,Moerdijk
52190,Zuid-Limburg,Stavropol,4602.0,357.0,"[1069947, 1069948, 1069949, 1069951, 1069960, ...",Zuid-Limburg,163.663380,Moerdijk


In [19]:
# Parameters
tonne_per_train=676
#cost_truck_average=0.138 # Data from (van der Meulen et al., 2023)
cost_truck_average=0.06 # Data from (Black et al., 2003). 
#cost_train_average=0.036 # Data from (van der Meulen et al., 2023)
cost_train_average=0.05 #  Data from (Black et al., 2003). 
#cost_haulage=cost_truck_average * 2
cost_haulage=cost_truck_average * 2
cost_handling=4
post_haulage_distance=50
extra_pre_dist=1.2

# Seperate costs for different type of goods (not used)
cost_truck_c=0.125
cost_truck_bb=0.153
cost_truck_b=0.138
cost_train_c=0.045
cost_train_bb=0.047
cost_train_b=0.016

In [20]:
average = filtered_df[
    [
        "Name_origin_region",
        "Name_destination_region",
        "Total_distance",
        "Traffic_flow_tons_2019",
        "min_distance_km",
        "closest_point",
        'Edge_path_E_road'
    ]
].copy()

In [21]:
# Calculate cost for each mode
average["price_truck"] = average["Total_distance"] *  tonne_per_train * cost_truck_average
average["price_train"] = average["Total_distance"] * tonne_per_train * cost_train_average
average["haulage"]= (tonne_per_train * post_haulage_distance * cost_haulage) + (tonne_per_train * average["min_distance_km"]*extra_pre_dist * cost_haulage)
average["container_handling"] = tonne_per_train*cost_handling
average["total_train_cost"] = average["price_train"] + average["haulage"] + average["container_handling"]

# Difference 
average["difference"] = (
    average["price_truck"] - average["total_train_cost"]
)



average = average.reset_index(drop=True)

# Show result
average.head(10)

,Name_origin_region,Name_destination_region,Total_distance,Traffic_flow_tons_2019,min_distance_km,closest_point,Edge_path_E_road,price_truck,price_train,haulage,container_handling,total_train_cost,difference
0,Oost-Groningen,Mittelburgenland,1145.0,17.0,246.267106,Amsterdam,"[1034974, 1008535, 1008552, 1000001, 1008851, ...",46441.20,38701.0,28028.625204,2704,69433.625204,-22992.425204
1,Oost-Groningen,Nordburgenland,1121.0,51.0,246.267106,Amsterdam,"[1008555, 2500448, 1008540, 1008817, 1008826, ...",45467.76,37889.8,28028.625204,2704,68622.425204,-23154.665204
2,Oost-Groningen,Sudburgenland,1243.0,17.0,246.267106,Amsterdam,"[1008563, 1310426, 1009011, 1035597, 1008566, ...",50416.08,42013.4,28028.625204,2704,72746.025204,-22329.945204
3,Oost-Groningen,Mostviertel-Eisenwurzen,1085.0,187.0,246.267106,Amsterdam,"[1008584, 1008595, 1008621, 1035056, 1008618, ...",44007.60,36673.0,28028.625204,2704,67405.625204,-23398.025204
4,Oost-Groningen,Niederosterreich-Sud,1164.0,221.0,246.267106,Amsterdam,"[1008654, 1008675, 1000017, 1035287, 1008678, ...",47211.84,39343.2,28028.625204,2704,70075.825204,-22863.985204
5,Oost-Groningen,Sankt Polten,1131.0,68.0,246.267106,Amsterdam,"[1034982, 1310502, 1008575, 1035027, 1008578, ...",45873.36,38227.8,28028.625204,2704,68960.425204,-23087.065204
6,Oost-Groningen,Waldviertel,1007.0,102.0,246.267106,Amsterdam,"[1008702, 1008725, 1008714, 1015464, 1010134, ...",40843.92,34036.6,28028.625204,2704,64769.225204,-23925.305204
7,Oost-Groningen,Weinviertel,1047.0,102.0,246.267106,Amsterdam,"[1024667, 1023214, 1310432, 1023215, 1023211, ...",42466.32,35388.6,28028.625204,2704,66121.225204,-23654.905204
8,Oost-Groningen,Wiener Umland/Nordteil,1044.0,663.0,246.267106,Amsterdam,"[1035558, 1035307, 1035306, 1008804, 1008801, ...",42344.64,35287.2,28028.625204,2704,66019.825204,-23675.185204
9,Oost-Groningen,Wiener Umland/Sudteil,1081.0,323.0,246.267106,Amsterdam,"[1024667, 1023214, 1310432, 1023215, 1023211, ...",43845.36,36537.8,28028.625204,2704,67270.425204,-23425.065204


In [22]:
# Total traffic flow tons where difference is positive
total_tons_positive = average.loc[
    average["difference"] > 0,
    "Traffic_flow_tons_2019"
].sum()

# Total traffic flow tons where difference is negative
total_tons_negative = average.loc[
    average["difference"] < 0,
    "Traffic_flow_tons_2019"
].sum()

print(f"Total trains where difference > 0: {total_tons_positive/676/365:,.0f}") # Amount of trains where train is cheaper than truck
print(f"Total trains where difference < 0: {total_tons_negative/676/365:,.0f}")

Total trains where difference > 0: 17
Total trains where difference < 0: 1,881


In [ ]:
#Only show O-D combinations where train is cheaper than truck
modal_shift = average.loc[
    average["difference"] > 0
].copy()
modal_shift

,Name_origin_region,Name_destination_region,Total_distance,Traffic_flow_tons_2019,min_distance_km,closest_point,Edge_path_E_road,price_truck,price_train,haulage,container_handling,total_train_cost,difference
1291,Oost-Groningen,Komi,4619.0,85.0,246.267106,Amsterdam,"[1031853, 1031925, 1031924, 1031641, 1032018, ...",187346.64,156122.2,28028.625204,2704,186854.825204,491.814796
1294,Oost-Groningen,Nenets,4557.0,85.0,246.267106,Amsterdam,"[1031853, 1031925, 1031924, 1031641, 1032018, ...",184831.92,154026.6,28028.625204,2704,184759.225204,72.694796
1297,Oost-Groningen,Amur,11333.0,85.0,246.267106,Amsterdam,"[1024667, 1023214, 1310432, 1023215, 1023211, ...",459666.48,383055.4,28028.625204,2704,413788.025204,45878.454796
1298,Oost-Groningen,Yevrey,11923.0,85.0,246.267106,Amsterdam,"[1024667, 1023214, 1310432, 1023215, 1023211, ...",483596.88,402997.4,28028.625204,2704,433730.025204,49866.854796
1299,Oost-Groningen,Kamchatka,14903.0,85.0,246.267106,Amsterdam,"[1024667, 1023214, 1310432, 1023215, 1023211, ...",604465.68,503721.4,28028.625204,2704,534454.025204,70011.654796
...,...,...,...,...,...,...,...,...,...,...,...,...,...
52187,Zuid-Limburg,Kabardin-Balkar,4562.0,357.0,163.663380,Moerdijk,"[1069947, 1069948, 1069949, 1069951, 1069960, ...",185034.72,154195.6,19987.648037,2704,176887.248037,8147.471963
52188,Zuid-Limburg,Karachay-Cherkess,4759.0,357.0,163.663380,Moerdijk,"[1069947, 1069948, 1069949, 1069951, 1069960, ...",193025.04,160854.2,19987.648037,2704,183545.848037,9479.191963
52189,Zuid-Limburg,North Ossetia,4480.0,357.0,163.663380,Moerdijk,"[1069947, 1069948, 1069949, 1069951, 1069960, ...",181708.80,151424.0,19987.648037,2704,174115.648037,7593.151963
52190,Zuid-Limburg,Stavropol,4602.0,357.0,163.663380,Moerdijk,"[1069947, 1069948, 1069949, 1069951, 1069960, ...",186657.12,155547.6,19987.648037,2704,178239.248037,8417.871963


In [ ]:
# Amount of trains per source
result = modal_shift.groupby('closest_point')['Traffic_flow_tons_2019'].sum().reset_index()
result["train_per_day"] = result['Traffic_flow_tons_2019'] / tonne_per_train / 365
result

,closest_point,Traffic_flow_tons_2019,train_per_day
0,Amsterdam,1134767.0,4.599039
1,Kijfhoek,1198194.0,4.856100
2,Moerdijk,1169277.0,4.738903
3,Sloe,725815.0,2.941619


In [ ]:
# edges which are located at the border
node_edges = {
    195926: [1021809, 1306842],
    195944: [1306275, 1306749],
    112601: [1018229, 1018218],
    117966: [1026610, 1026609, 1026599],
    113494: [1025018, 1024983, 1019618],
    113468: [1024882, 1019602],
    181006: [1201007, 1024767]
}

node_sums = {}

# Total trains for each border
for node, edges in node_edges.items():
    filtered = modal_shift[
        modal_shift['Edge_path_E_road'].apply(
            lambda x: isinstance(x, str) and any(str(edge) in x for edge in edges)
        )
    ]
    node_sums[node] = filtered['Traffic_flow_tons_2019'].sum()

for node, total in node_sums.items():
    print(f"Node {node}: Total traffic = {total} tons and {total/676/365:,.0f} trains per day")